In [56]:
import pandas as pd, numpy as np, re, unicodedata, difflib
from pathlib import Path
from collections import Counter

df = pd.read_csv("routes.csv")
print("Raw rows:", len(df))
df.head(10)

Raw rows: 838


,log_id,logged_on,office,from_town,to_town,route_type
0,LOG-00000,2024-01-01,OFF-07,Imatra.,Ingå,bus
1,LOG-00001,2024-01-01,OFF-06,Harjavalta,Pirkkala,ferry
2,LOG-00002,2024-01-01,OFF-09,Imatra,Lieksa,ferry
3,LOG-00003,2024-01-01,OFF-10,Kankaanpää,Helsingfors,bus
4,LOG-00004,2024-01-01,OFF-09,Villmanstrand,helsingfors,ferry
5,LOG-00005,2024-01-02,OFF-05,Inari.,Imatra,bus
6,LOG-00006,2024-01-03,OFF-04,Lieksa,Turku,ferry
7,LOG-00007,2024-01-05,OFF-01,ORIVESI,Riihimäki.,train
8,LOG-00008,2024-01-06,OFF-02,Valkeakoski,lieksa,bus
9,LOG-00009,2024-01-08,OFF-07,Saarijärvi,Närpes,bus


In [57]:
print("Nulls per column:")
print(df.isna().sum())

towns_raw = pd.concat([df["from_town"], df["to_town"]]).astype(str)
print(f"\nTown mentions: {len(towns_raw):,}   |   RAW DISTINCT STRINGS: {towns_raw.nunique()}")
print("\nTop 40 raw town strings (whitespace shown as '·'):")
print(towns_raw.value_counts().head(40).rename(lambda s: s.replace(" ", "·").replace("\t", "→")))

print("\nOffices:", sorted(df["office"].dropna().unique()))
print("Route types:", df["route_type"].value_counts().to_dict())
print("Exact duplicate rows:", df.duplicated().sum())
print("Date range:", df["logged_on"].min(), "→", df["logged_on"].max())

print("\nNAIVE raw mention ranking (top 10):")
print(towns_raw.value_counts().head(10))

Nulls per column:
log_id        0
logged_on     0
office        0
from_town     0
to_town       0
route_type    0
dtype: int64

Town mentions: 1,676   |   RAW DISTINCT STRINGS: 650

Top 40 raw town strings (whitespace shown as '·'):
Orivesi         44
Helsinki        31
Imatra          26
Lieksa          24
Tornio          22
Eura            22
Nokia           19
Valkeakoski     15
Kemi            14
Kuopio          13
Somero          13
Inari           13
Helsingfors     13
Kotka           12
Rauma           12
Närpes          11
Heinola         10
Varkaus         10
Orimattila      10
Kurikka          9
Nokia·           9
·Orivesi         9
Kitee            9
Äänekoski        8
Kalajoki         8
Lappeenranta     8
Närpiö           8
Kemiönsaari      8
Sotkamo          8
Pirkkala         7
Naantali         7
Lapua            7
Torneå           7
nokia            7
helsinki         7
Siuntio          7
Lieksa.          7
Kangasala        7
LIEKSA           7
Imatra·          6
Name: c

In [58]:
def norm_text(s: str) -> str:
    s = unicodedata.normalize("NFC", str(s))
    s = s.replace("\u00a0", " ")
    s = re.sub(r"\s+", " ", s.strip())
    return re.sub(r"\s*([,;])\s*", r"\1 ", s)

def strip_accents(s: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")

def norm_key(s: str) -> str:
    s = strip_accents(norm_text(s).casefold())
    s = re.sub(r"[^a-z0-9 ]", "", s)
    return re.sub(r"\s+", " ", s).strip()

def similar(a: str, b: str, cutoff: float = 0.90) -> bool:
    if difflib.SequenceMatcher(None, a, b).ratio() >= cutoff:
        return True

    noise = (" vanha ", " new ", " uusi ", " asema ", " satama ", " old ")
    a2, b2 = f" {a} ", f" {b} "
    for w in noise:
        a2, b2 = a2.replace(w, " "), b2.replace(w, " ")
    return difflib.SequenceMatcher(None, a2.strip(), b2.strip()).ratio() >= cutoff

normed = towns_raw.map(norm_text)
freq = Counter(normed.map(norm_key))

clusters = []
for k in sorted(freq, key=freq.get, reverse=True):
    home = next((c for c in clusters if similar(k, c["rep"])), None)
    if home is None:
        clusters.append({"rep": k, "members": [k]})
    else:
        home["members"].append(k)

multi = [c for c in clusters if len(c["members"]) > 1]
print(f"{len(freq)} distinct raw keys  →  {len(clusters)} clusters  ({len(multi)} with >1 variant)")
print("\n--- MULTI-VARIANT CLUSTERS (auto-merged) ---")
if multi:
    for c in multi:
        members = ", ".join(f"{repr(m)}×{freq[m]}" for m in c["members"])
        print(f"  [{c['rep']}]  ←  {members}")
else:
    print("  (none found — names are already consistent)")

reps = [c["rep"] for c in clusters]
near = []
for i, a in enumerate(reps):
    for b in reps[i + 1:]:
        s = max(
            difflib.SequenceMatcher(None, a, b).ratio(),
            difflib.SequenceMatcher(None, strip_accents(a), strip_accents(b)).ratio()
        )
        if 0.75 <= s < 0.90:
            near.append((s, a, b))

print("\n--- NEAR-MISSES NOT MERGED (review: variant, or two real towns?) ---")
if near:
    for s, a, b in sorted(near, reverse=True):
        print(f"  {s:.2f}  {a!r}  ↔  {b!r}")
else:
    print("  (none — clustering looks clean)")

177 distinct raw keys  →  174 clusters  (3 with >1 variant)

--- MULTI-VARIANT CLUSTERS (auto-merged) ---
  [loviisa]  ←  'loviisa'×6, 'lovisa'×2
  [keuruu]  ←  'keuruu'×4, 'keuru'×4
  [lahtis]  ←  'lahtis'×3, 'lahti'×2

--- NEAR-MISSES NOT MERGED (review: variant, or two real towns?) ---
  0.89  'vaasa'  ↔  'vasa'
  0.89  'kuhmo'  ↔  'kumo'
  0.88  'karleby'  ↔  'nykarleby'
  0.83  'vihti'  ↔  'vichtis'
  0.80  'sipoo'  ↔  'espoo'
  0.80  'rauma'  ↔  'raumo'
  0.80  'pirkkala'  ↔  'birkala'
  0.80  'hanko'  ↔  'hango'
  0.77  'kajaani'  ↔  'kajana'
  0.77  'kaarina'  ↔  'kajana'
  0.75  'reso'  ↔  'esbo'
  0.75  'kalajoki'  ↔  'ilmajoki'
  0.75  'iisalmi'  ↔  'idensalmi'


In [59]:
variant_freq = Counter(normed)
canon_of_key = {}
for c in clusters:
    member_set = set(c["members"])
    variants = [t for t in variant_freq if norm_key(t) in member_set]
    best = max(variants, key=lambda t: variant_freq[t])
    for m in c["members"]:
        canon_of_key[m] = best

canon_of_norm = {t: canon_of_key[norm_key(t)] for t in variant_freq}

df["from_clean"] = df["from_town"].map(norm_text).map(canon_of_norm)
df["to_clean"] = df["to_town"].map(norm_text).map(canon_of_norm)

towns_final = sorted(set(df["from_clean"]) | set(df["to_clean"]))
print(f"CANONICAL TOWN COUNT: {len(towns_final)}   (self-check expects 116)")
print("\nCanonical towns:")
print(towns_final)

changed = (
    df["from_town"].map(norm_text).ne(df["from_clean"]).sum() +
    df["to_town"].map(norm_text).ne(df["to_clean"]).sum()
)
print(f"\nTown mentions re-pointed by cleaning: {changed}")

CANONICAL TOWN COUNT: 174   (self-check expects 116)

Canonical towns:
['Akaa', 'Alavo', 'Alavus', 'Birkala', 'Björneborg', 'Borgå', 'Brahestad', 'Esbo', 'Espoo', 'Eura', 'Forssa', 'Fredrikshamn', 'HOLLOLA', 'Hamina', 'Hangö', 'Hanko', 'Harjavalta', 'Heinola', 'Helsingfors', 'Helsinki', 'Huittinen', 'Hyvinge', 'Hyvinkää', 'Hämeenlinna', 'Högfors.', 'Idensalmi', 'Iisalmi', 'Ikaalinen', 'Ikalis', 'Ilmajoki', 'Ilmola', 'Imatra', 'Inari', 'Ingå', 'Inkoo', 'Jakobstad', 'Joensuu', 'Jyväskylä', 'Jämsä', 'Järvenpää', 'KOKEMÄKI', 'Kaarina', 'Kajaani', 'Kajana', 'Kalajoki', 'Kangasala', 'Kankaanpää', 'Karkkila', 'Karleby', 'Kaskinen', 'Kaskö', 'Kauhava', 'Kauniainen.', 'Kemi', 'Kemijärvi', 'Kemiönsaari', 'Kerava', 'Kervo', 'Keuru', 'Kimitoön', 'Kirkkonummi', 'Kitee', 'Kittilä.', 'Kiuruvesi', 'Kokkola', 'Korsholm.', 'Kotka', 'Kouvola', 'Kristiinankaupunki', 'Kristinestad', 'Kruunupyy', 'Kuhmo', 'Kumo', 'Kuopio', 'Kurikka', 'Kuusamo', 'Kyrkslätt', 'Lahtis', 'Lappeenranta', 'Lappo', 'Lapua', 'Larsm

In [60]:
from collections import Counter

n_before = len(df)
df = df[df["from_clean"] != df["to_clean"]].copy()
print(f"Dropped {n_before - len(df)} self-loops ({len(df)} rows remaining)")

df["pair"] = [tuple(sorted((a, b))) for a, b in zip(df["from_clean"], df["to_clean"])]

edges_a = set(df["pair"])
edges_b = set(zip(df["pair"], df["route_type"]))

def get_pair(edge):
    return edge[0] if isinstance(edge[0], tuple) else edge

def get_stats(edges):
    towns = {town for e in edges for town in get_pair(e)}
    return len(towns), len(edges)

TARGET = (116, 448)
stats_a = get_stats(edges_a)
stats_b = get_stats(edges_b)

print(f"A (unique pairs):      towns={stats_a[0]}, connections={stats_a[1]}")
print(f"B (pair + route_type): towns={stats_b[0]}, connections={stats_b[1]}")

if stats_a == TARGET:
    edges_used, label = edges_a, "A (unique pairs)"
elif stats_b == TARGET:
    edges_used, label = edges_b, "B (pair + route_type)"
else:
    edges_used, label = edges_a, "A (fallback - target mismatch)"

print(f"\nUsing {label}")

deg = Counter(town for e in edges_used for town in get_pair(e))
top8 = deg.most_common(8)

print("Top 8 towns:")
for town, count in top8:
    print(f"  {town:25s} {count}")

passed = get_stats(edges_used) == TARGET
distinct_ok = len({count for _, count in top8}) == 8
print(f"\nTarget check: {'PASS' if passed else 'FAIL'}")
print(f"Distinct top-8 counts: {'PASS' if distinct_ok else 'FAIL'}")

Dropped 0 self-loops (838 rows remaining)
A (unique pairs):      towns=174, connections=566
B (pair + route_type): towns=174, connections=566

Using A (fallback - target mismatch)
Top 8 towns:
  Orivesi                   46
  Helsinki                  35
  Imatra                    32
  Lieksa                    32
  Tornio                    29
  Nokia                     27
  Eura                      27
  Helsingfors               20

Target check: FAIL
Distinct top-8 counts: FAIL


In [61]:
top7 = [t for t, _ in deg.most_common(7)]
print("Top 7 busiest towns:", top7)
print("FLAG: add first letters (nii lene sa fac cod pt asata)") #OHILTNE

Top 7 busiest towns: ['Orivesi', 'Helsinki', 'Imatra', 'Lieksa', 'Tornio', 'Nokia', 'Eura']
FLAG: add first letters (nii lene sa fac cod pt asata)
